# Spatial Data Cleaning: HUC-12 Watershed Crosswalk

Cleans the raw NHDPlus / Watershed Boundary Dataset (WBD) HUC-12 polygon layer
and spatially joins it against the EPA water-quality monitoring stations to
produce a **station → watershed** crosswalk keyed on
`MonitoringLocationIdentifier`. This is the spatial glue needed to attach
HUC-12-scale covariates (CDL land-cover fractions, INRS BMP adoption) to
individual stations in a future land-use/BMP merge step.

**Input:**
- `data/spatial/01_raw/nhdplus/wbd-huc12-iowa/WBDSnapshot_Iowa.shp` (1,714 HUC-12 watershed polygons, NAD83 / EPSG:4269)
- `data/tabular/02_clean/water-quality/epa-stations-clean.csv` (1,666 station points)

**Output:** `data/spatial/02_clean/nhdplus/wbd-huc12-station-crosswalk-clean.csv`

**Cleaning steps:**

1. Load the raw WBD polygons and reproject to WGS84 (EPSG:4326) to match the
   station lat/lon; confirm every polygon geometry is valid.
2. Reduce to the join-relevant columns and rename to the snake_case id
   convention used elsewhere (`huc8_code`, `huc10_code`, `huc12_code`,
   `huc12_name`, `huc12_acres`). Unlike `cdl-huc12-fractions-clean.ipynb`,
   the shapefile's HUC columns are stored as fixed-width strings, so no
   zero-pad fix is needed here — we just assert the expected digit lengths.
3. Confirm `huc12_code` is already unique across polygons (no dissolve needed).
4. Load the cleaned station table and spatially join each station point to
   the watershed polygon it falls **within**.
5. **QA** — assert every station matched exactly one polygon (no unmatched
   points, no boundary-overlap fan-out), and cross-check the derived
   `huc8_code` against the station table's own (non-zero-padded)
   `HUCEightDigitCode` column.
6. Select/order columns, sort by `MonitoringLocationIdentifier`, and write
   the tidy crosswalk to `02_clean`.

In [1]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

In [2]:
def find_repo_root(start: Path | None = None) -> Path:
    """Walk upward until we find the repo's data/tabular directory.

    Notebooks have no __file__, and the kernel's working directory varies, so
    resolving paths relative to a fixed number of "../" is fragile. Searching
    upward for a sentinel makes the notebook runnable from anywhere.
    """
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "data" / "tabular").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate repo root containing data/tabular/")


REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / "data" / "spatial" / "01_raw" / "nhdplus"
STATIONS_PATH = REPO_ROOT / "data" / "tabular" / "02_clean" / "water-quality" / "epa-stations-clean.csv"
CLEAN_DIR = REPO_ROOT / "data" / "spatial" / "02_clean" / "nhdplus"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
print("Repo root:  ", REPO_ROOT)
print("Raw dir:    ", RAW_DIR)
print("Stations:   ", STATIONS_PATH)
print("Clean dir:  ", CLEAN_DIR)

Repo root:   /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction
Raw dir:     /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/spatial/01_raw/nhdplus
Stations:    /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/tabular/02_clean/water-quality/epa-stations-clean.csv
Clean dir:   /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/spatial/02_clean/nhdplus


## Step 1 — Load & reproject the watershed polygons

The raw WBD snapshot is in NAD83 geographic coordinates (EPSG:4269). The
station table's `LatitudeMeasure`/`LongitudeMeasure` mix NAD83, WGS84, and a
few NAD27/unknown datums (per the raw `HorizontalCoordinateReferenceSystemDatumName`
column) — those datums differ by at most a few meters in Iowa, far under
HUC-12 polygon scale, so we reproject the polygons to WGS84 (EPSG:4326) and
treat the station coordinates as WGS84 for the join.

In [3]:
huc12 = gpd.read_file(RAW_DIR / "wbd-huc12-iowa" / "WBDSnapshot_Iowa.shp")
print(f"Loaded {len(huc12):,} polygons, CRS = {huc12.crs}")

huc12 = huc12.to_crs(4326)
n_invalid = (~huc12.geometry.is_valid).sum()
print(f"Invalid geometries: {n_invalid}")
assert n_invalid == 0, "invalid polygon geometry found"

Loaded 1,714 polygons, CRS = EPSG:4269


Invalid geometries: 0


## Step 2 — Select & rename columns

Keep only the identifying/join columns and rename to the snake_case `_code`
convention used by the other cleaners (e.g. `cdl-huc12-fractions-clean.ipynb`).
The shapefile stores `HUC_8`/`HUC_10`/`HUC_12` as fixed-width strings (unlike
the CSV-sourced CDL table, where reading the id as an integer stripped the
leading zero), so we only need to assert the digit lengths, not fix them.

In [4]:
COLUMN_MAP = {
    "HUC_8": "huc8_code",
    "HUC_10": "huc10_code",
    "HUC_12": "huc12_code",
    "HU_12_NAME": "huc12_name",
    "ACRES": "huc12_acres",
}
huc12 = huc12[list(COLUMN_MAP) + ["geometry"]].rename(columns=COLUMN_MAP)

lengths = {
    c: sorted(huc12[c].str.len().unique().tolist())
    for c in ["huc8_code", "huc10_code", "huc12_code"]
}
print("Observed digit lengths:", lengths)
assert lengths["huc8_code"] == [8]
assert lengths["huc10_code"] == [10]
assert lengths["huc12_code"] == [12]

Observed digit lengths: {'huc8_code': [8], 'huc10_code': [10], 'huc12_code': [12]}


## Step 3 — Confirm polygon key uniqueness

`huc12_code` should uniquely identify a polygon (no split/multi-part
watersheds in this extract). If this ever fails, the fix is to `dissolve()`
on `huc12_code` before the join.

In [5]:
dup_polygons = huc12["huc12_code"].duplicated().sum()
print(f"Duplicate huc12_code polygons: {dup_polygons}")
assert dup_polygons == 0, "duplicate HUC-12 polygon; dissolve needed"

Duplicate huc12_code polygons: 0


## Step 4 — Load stations & spatial join

Build a point GeoDataFrame from the cleaned station table and join each
point to the watershed polygon it falls **within**.

In [6]:
stations = pd.read_csv(STATIONS_PATH)
print(f"Loaded {len(stations):,} stations")

points = gpd.GeoDataFrame(
    stations,
    geometry=gpd.points_from_xy(stations["LongitudeMeasure"], stations["LatitudeMeasure"]),
    crs=4326,
)

joined = gpd.sjoin(
    points,
    huc12,
    how="left",
    predicate="within",
).drop(columns="index_right")
print(f"Joined rows: {len(joined):,}")

Loaded 1,666 stations
Joined rows: 1,666


## Step 5 — QA the join

Every station should match **exactly one** polygon: no unmatched points
(coordinates outside all HUC-12 boundaries) and no fan-out (a point sitting
exactly on a shared boundary matching more than one polygon). As an
independent check, the HUC-8 prefix implied by the spatial join should agree
with the station table's own `HUCEightDigitCode` column, which was sourced
from WQX metadata rather than derived geometrically.

In [7]:
n_unmatched = joined["huc12_code"].isna().sum()
print(f"Stations with no matching polygon: {n_unmatched}")
assert n_unmatched == 0, "station fell outside all HUC-12 polygons"

n_fanout = len(joined) - len(stations)
print(f"Extra rows from boundary overlap: {n_fanout}")
assert n_fanout == 0, "station matched more than one polygon"

existing_huc8 = (
    stations.set_index("MonitoringLocationIdentifier")["HUCEightDigitCode"]
    .astype(str)
    .str.zfill(8)
)
derived_huc8 = joined.set_index("MonitoringLocationIdentifier")["huc8_code"]
mismatch = (existing_huc8 != derived_huc8).sum()
print(
    f"huc8_code mismatches vs. station table's own HUCEightDigitCode: "
    f"{mismatch} / {len(stations)}"
)
assert mismatch == 0, "spatially-derived HUC-8 disagrees with WQX metadata"

Stations with no matching polygon: 0
Extra rows from boundary overlap: 0
huc8_code mismatches vs. station table's own HUCEightDigitCode: 0 / 1666


## Step 6 — Select, sort & write

Reorder columns to lead with the station key, sort, and write the tidy
crosswalk to `02_clean`.

In [8]:
OUTPUT_COLS = [
    "MonitoringLocationIdentifier",
    "huc12_code",
    "huc12_name",
    "huc10_code",
    "huc8_code",
    "huc12_acres",
]
crosswalk = (
    joined[OUTPUT_COLS]
    .sort_values("MonitoringLocationIdentifier")
    .reset_index(drop=True)
)

dup_stations = crosswalk["MonitoringLocationIdentifier"].duplicated().sum()
assert dup_stations == 0, "duplicate station in crosswalk"

out_path = CLEAN_DIR / "wbd-huc12-station-crosswalk-clean.csv"
crosswalk.to_csv(out_path, index=False)
print(f"Wrote {len(crosswalk):,} rows × {crosswalk.shape[1]} cols to:")
print(" ", out_path.relative_to(REPO_ROOT))
crosswalk.head()

Wrote 1,666 rows × 6 cols to:
  data/spatial/02_clean/nhdplus/wbd-huc12-station-crosswalk-clean.csv


,MonitoringLocationIdentifier,huc12_code,huc12_name,huc10_code,huc8_code,huc12_acres
0,11NPSWRD_WQX-HTLN_EFMO_DOUS1,070600010906,Lower Yellow River,0706000109,07060001,30152.105056
1,11NPSWRD_WQX-HTLN_HEHO_HOOV1,070802060702,West Branch Wapsinonoc Creek,0708020607,07080206,36037.943023
2,21IOWA_WQX-10030001,070600020602,Paint Creek-Upper Iowa River,0706000206,07060002,23677.702695
3,21IOWA_WQX-10030002,070600010906,Lower Yellow River,0706000109,07060001,30152.105056
4,21IOWA_WQX-10040002,102802010404,Snyder Branch-Chariton River,1028020104,10280201,19356.516649
